# Prepared Dataset and Methods Plan

**Project:** Mining EV Registration Patterns in Washington State  
**Authors:** Afsana Tasnim Juha, Mamata Joshi, Prasanna Thapa

## 1. Load and check the sample


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display


DATA_PATH = Path("data/raw/ev_sample_50k.csv")

print("Current working folder:", Path.cwd())
print("Dataset found:", DATA_PATH.is_file())

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "Cannot find data/raw/ev_sample_50k.csv. "
        "Check the notebook's working folder and dataset location."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)


required_columns = [
    "DOL Vehicle ID",
    "State",
    "Postal Code",
    "Make",
    "Model",
    "Model Year",
    "Electric Vehicle Type",
    "Electric Range"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

print("\nDataset shape:", df.shape)
print("Missing required columns:", missing_columns)
print("Exact duplicate rows:", df.duplicated().sum())

assert df.shape == (50000, 16), (
    "Dataset dimensions differ from the reviewed working sample."
)
assert not missing_columns, (
    f"Required columns are missing: {missing_columns}"
)
assert df["DOL Vehicle ID"].notna().all(), (
    "Some vehicle IDs are missing."
)
assert df["DOL Vehicle ID"].is_unique, (
    "Some vehicle IDs are repeated."
)
assert df.duplicated().sum() == 0, (
    "Exact duplicate rows were found."
)

print("\nColumn types and missing values:")
display(pd.DataFrame({
    "Data_Type": df.dtypes.astype(str),
    "Missing_Count": df.isna().sum()
}))

print("\nFirst five records:")
display(df.head())



Current working folder: c:\Users\Asus\Documents\Data Mining Project F26
Dataset found: True

Dataset shape: (50000, 16)
Missing required columns: []
Exact duplicate rows: 0

Column types and missing values:


,Data_Type,Missing_Count
VIN (1-10),str,0
County,str,2
City,str,2
State,str,0
Postal Code,float64,2
Model Year,int64,0
Make,str,0
Model,str,0
Electric Vehicle Type,str,0
Clean Alternative Fuel Vehicle (CAFV) Eligibility,str,0



First five records:


,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 GEOID
0,JTJABABB3R,Thurston,Lacey,WA,98516.0,2024,LEXUS,RZ,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,22.0,272203270,POINT (-122.78598 47.08476),PUGET SOUND ENERGY INC,5.306701e+10
1,7YAKPDDC6S,King,Seattle,WA,98103.0,2025,HYUNDAI,IONIQ 5,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,36.0,286868704,POINT (-122.34301 47.65918),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303300e+10
2,1V2WSPE85S,Clark,Vancouver,WA,98683.0,2025,VOLKSWAGEN,ID.4,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,17.0,285893233,POINT (-122.49212 45.60365),BONNEVILLE POWER ADMINISTRATION||PUD NO 1 OF C...,5.301104e+10
3,7PDSGABA6P,King,Sammamish,WA,98074.0,2023,RIVIAN,R1S,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,41.0,252551757,POINT (-122.03133 47.62858),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),5.303303e+10
4,7SAXCBE57R,Snohomish,Mill Creek,WA,98012.0,2024,TESLA,MODEL X,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,44.0,269998305,POINT (-122.20105 47.84423),PUGET SOUND ENERGY INC,5.306105e+10


## 2. Apply the preprocessing rules

- Keep ZIP codes as five-character strings.
- Keep original range values; treat zero and missing values as unavailable.
- Use positive values for usable range.
- Keep Washington records with valid five-digit ZIP codes.
- Add BEV and PHEV indicator columns.
Positive but unusual ranges stay for later anomaly review.

In [2]:

df_clean = df.copy()


df_clean["Postal Code"] = (
    df_clean["Postal Code"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)


df_clean["Electric Range Usable"] = (
    df_clean["Electric Range"]
    .where(df_clean["Electric Range"] > 0)
)


is_wa = df_clean["State"].eq("WA")

valid_zip = (
    df_clean["Postal Code"]
    .str.fullmatch(r"\d{5}")
    .fillna(False)
)

df_wa = df_clean.loc[is_wa & valid_zip].copy()


expected_types = [
    "Battery Electric Vehicle (BEV)",
    "Plug-in Hybrid Electric Vehicle (PHEV)"
]

assert df_wa["Electric Vehicle Type"].isin(expected_types).all(), (
    "Unexpected or missing vehicle types were found."
)

df_wa["Is BEV"] = (
    df_wa["Electric Vehicle Type"]
    .eq("Battery Electric Vehicle (BEV)")
    .astype(int)
)

df_wa["Is PHEV"] = (
    df_wa["Electric Vehicle Type"]
    .eq("Plug-in Hybrid Electric Vehicle (PHEV)")
    .astype(int)
)


assert len(df_clean) == len(df)
assert df_wa["State"].eq("WA").all()
assert df_wa["Postal Code"].str.fullmatch(r"\d{5}").all()
assert (df_wa["Is BEV"] + df_wa["Is PHEV"]).eq(1).all()
assert df_wa["Electric Range Usable"].dropna().gt(0).all()
assert df_clean["Electric Range"].equals(df["Electric Range"])


summary = pd.Series({
    "Original sample records": len(df),
    "Non-Washington records excluded": int((~is_wa).sum()),
    "Washington records with invalid/missing ZIP excluded":
        int((is_wa & ~valid_zip).sum()),
    "Washington records retained": len(df_wa),
    "Unique Washington ZIP codes": df_wa["Postal Code"].nunique(),
    "BEV records": int(df_wa["Is BEV"].sum()),
    "PHEV records": int(df_wa["Is PHEV"].sum()),
    "Positive recorded ranges":
        int(df_wa["Electric Range Usable"].notna().sum()),
    "Unavailable ranges":
        int(df_wa["Electric Range Usable"].isna().sum())
}, name="Count")

print(summary.to_string())

print("\nRange availability by vehicle type:")
range_summary = df_wa.groupby("Electric Vehicle Type").agg(
    Vehicle_Count=("DOL Vehicle ID", "size"),
    Positive_Range_Count=("Electric Range Usable", "count")
)

range_summary["Range_Available_Percent"] = (
    range_summary["Positive_Range_Count"]
    / range_summary["Vehicle_Count"] * 100
)

display(range_summary.round(2))

print("\nPrepared record preview:")
display(df_wa[[
    "DOL Vehicle ID",
    "Postal Code",
    "Electric Vehicle Type",
    "Electric Range",
    "Electric Range Usable",
    "Is BEV",
    "Is PHEV"
]].head())



Original sample records                                 50000
Non-Washington records excluded                           127
Washington records with invalid/missing ZIP excluded        0
Washington records retained                             49873
Unique Washington ZIP codes                               509
BEV records                                             40168
PHEV records                                             9705
Positive recorded ranges                                17202
Unavailable ranges                                      32671

Range availability by vehicle type:


,Vehicle_Count,Positive_Range_Count,Range_Available_Percent
Electric Vehicle Type,,,
Battery Electric Vehicle (BEV),40168,7504,18.68
Plug-in Hybrid Electric Vehicle (PHEV),9705,9698,99.93



Prepared record preview:


,DOL Vehicle ID,Postal Code,Electric Vehicle Type,Electric Range,Electric Range Usable,Is BEV,Is PHEV
0,272203270,98516,Battery Electric Vehicle (BEV),0.0,NaN,1,0
1,286868704,98103,Battery Electric Vehicle (BEV),0.0,NaN,1,0
2,285893233,98683,Battery Electric Vehicle (BEV),0.0,NaN,1,0
3,252551757,98074,Battery Electric Vehicle (BEV),0.0,NaN,1,0
4,269998305,98012,Battery Electric Vehicle (BEV),0.0,NaN,1,0


## 3. Prepare ZIP-Level Fleet Profiles

Group vehicles by ZIP and calculate vehicle counts, BEV/PHEV counts, average model year, and range summaries.

Use ZIP codes with at least 30 sampled vehicles for clustering. Keep the full summary so excluded ZIPs remain visible. Range fields are descriptive only. These results describe the sample, not total EV ownership or adoption rates.

In [3]:

MIN_ZIP_VEHICLES = 30


zip_summary = df_wa.groupby("Postal Code").agg(
    EV_Count=("DOL Vehicle ID", "size"),
    BEV_Count=("Is BEV", "sum"),
    PHEV_Count=("Is PHEV", "sum"),
    Avg_Model_Year=("Model Year", "mean"),
    Avg_Electric_Range=("Electric Range Usable", "mean"),
    Valid_Range_Count=("Electric Range Usable", "count")
).reset_index()


zip_summary["BEV_Percentage"] = (
    zip_summary["BEV_Count"] / zip_summary["EV_Count"] * 100
)

zip_summary["Range_Available_Percent"] = (
    zip_summary["Valid_Range_Count"]
    / zip_summary["EV_Count"] * 100
)


zip_summary["Included_In_Clustering"] = (
    zip_summary["EV_Count"] >= MIN_ZIP_VEHICLES
)

zip_prepared = zip_summary.loc[
    zip_summary["Included_In_Clustering"]
].copy().reset_index(drop=True)


assert zip_summary["Postal Code"].is_unique
assert zip_summary["EV_Count"].sum() == len(df_wa)
assert (
    zip_summary["BEV_Count"] + zip_summary["PHEV_Count"]
).eq(zip_summary["EV_Count"]).all()

assert zip_summary["Valid_Range_Count"].le(
    zip_summary["EV_Count"]
).all()

assert zip_summary["BEV_Percentage"].between(0, 100).all()
assert zip_summary["Range_Available_Percent"].between(0, 100).all()
assert zip_prepared["EV_Count"].ge(MIN_ZIP_VEHICLES).all()

assert not zip_prepared[
    ["EV_Count", "BEV_Percentage", "Avg_Model_Year"]
].isna().any().any()


retained_vehicles = int(zip_prepared["EV_Count"].sum())
coverage_percent = retained_vehicles / len(df_wa) * 100

print("All ZIP codes:", len(zip_summary))
print("ZIP codes retained for clustering:", len(zip_prepared))
print("ZIP codes excluded from primary clustering:",
      len(zip_summary) - len(zip_prepared))
print("Vehicles represented by retained ZIPs:", retained_vehicles)
print(f"Washington sample coverage: {coverage_percent:.2f}%")
print(
    "Retained ZIPs without positive-range observations:",
    int(zip_prepared["Valid_Range_Count"].eq(0).sum())
)

print("\nFirst five retained ZIP profiles:")
display(zip_prepared.head().round(2))

print("\nRetained ZIP profile statistics:")
display(
    zip_prepared[[
        "EV_Count",
        "BEV_Percentage",
        "Avg_Model_Year",
        "Avg_Electric_Range",
        "Valid_Range_Count"
    ]].describe().round(2)
)



All ZIP codes: 509
ZIP codes retained for clustering: 249
ZIP codes excluded from primary clustering: 260
Vehicles represented by retained ZIPs: 47870
Washington sample coverage: 95.98%
Retained ZIPs without positive-range observations: 0

First five retained ZIP profiles:


,Postal Code,EV_Count,BEV_Count,PHEV_Count,Avg_Model_Year,Avg_Electric_Range,Valid_Range_Count,BEV_Percentage,Range_Available_Percent,Included_In_Clustering
0,98001,225,196,29,2022.65,118.11,57,87.11,25.33,True
1,98002,92,64,28,2021.87,87.15,46,69.57,50.00,True
2,98003,193,157,36,2022.34,99.90,63,81.35,32.64,True
3,98004,662,583,79,2022.76,135.40,177,88.07,26.74,True
4,98005,355,307,48,2023.18,118.24,93,86.48,26.20,True



Retained ZIP profile statistics:


,EV_Count,BEV_Percentage,Avg_Model_Year,Avg_Electric_Range,Valid_Range_Count
count,249.00,249.00,249.00,249.00,249.00
mean,192.25,78.83,2022.24,104.24,65.78
std,170.70,6.86,0.54,20.80,51.04
min,30.00,39.51,2020.73,41.84,2.00
25%,71.00,75.34,2021.95,92.52,29.00
50%,137.00,79.24,2022.26,102.56,54.00
75%,251.00,83.65,2022.57,115.76,88.00
max,1208.00,98.39,2024.35,186.40,305.00


## 4. Finalize and Scale Clustering Features

Use three features: Log_EV_Count, BEV_Percentage, and Avg_Model_Year. Apply log1p to counts, then standardize the features using eligible ZIP codes only.

ZIP code is an ID, not a feature. Range summaries are kept for interpretation, not clustering. This version adds the 30-vehicle cutoff and log transform.

In [4]:
import numpy as np
from sklearn.preprocessing import StandardScaler

zip_prepared["Log_EV_Count"] = np.log1p(
    zip_prepared["EV_Count"]
)

cluster_features = [
    "Log_EV_Count",
    "BEV_Percentage",
    "Avg_Model_Year"
]


X_cluster = zip_prepared[cluster_features].copy()

assert np.isfinite(X_cluster.to_numpy()).all(), (
    "Clustering features contain missing or infinite values."
)
assert X_cluster.std(ddof=0).gt(0).all(), (
    "A clustering feature has no variation."
)


cluster_scaler = StandardScaler()

X_cluster_scaled = pd.DataFrame(
    cluster_scaler.fit_transform(X_cluster),
    columns=cluster_features,
    index=zip_prepared.index
)


zip_scaled = X_cluster_scaled.copy()
zip_scaled.insert(
    0, "Postal Code", zip_prepared["Postal Code"]
)


assert len(X_cluster_scaled) == len(zip_prepared)
assert np.isfinite(X_cluster_scaled.to_numpy()).all()

assert np.allclose(
    X_cluster_scaled.mean().to_numpy(),
    0,
    atol=1e-10
)

assert np.allclose(
    X_cluster_scaled.std(ddof=0).to_numpy(),
    1,
    atol=1e-10
)

assert zip_scaled["Postal Code"].equals(
    zip_prepared["Postal Code"]
)


assert np.allclose(
    cluster_scaler.inverse_transform(X_cluster_scaled),
    X_cluster.to_numpy()
)

print("Clustering feature matrix shape:", X_cluster_scaled.shape)
print("Table with ZIP identifiers shape:", zip_scaled.shape)

print("\nScaling verification:")
display(pd.DataFrame({
    "Mean_After_Scaling": X_cluster_scaled.mean(),
    "Std_After_Scaling": X_cluster_scaled.std(ddof=0)
}).round(6))

print("\nOriginal feature preview:")
display(
    zip_prepared[
        ["Postal Code", "EV_Count"] + cluster_features
    ].head().round(4)
)

print("\nScaled feature preview:")
display(zip_scaled.head().round(4))



Clustering feature matrix shape: (249, 3)
Table with ZIP identifiers shape: (249, 4)

Scaling verification:


,Mean_After_Scaling,Std_After_Scaling
Log_EV_Count,0.0,1.0
BEV_Percentage,0.0,1.0
Avg_Model_Year,0.0,1.0



Original feature preview:


,Postal Code,EV_Count,Log_EV_Count,BEV_Percentage,Avg_Model_Year
0,98001,225,5.4205,87.1111,2022.6489
1,98002,92,4.5326,69.5652,2021.8696
2,98003,193,5.2679,81.3472,2022.3368
3,98004,662,6.4968,88.0665,2022.7583
4,98005,355,5.8749,86.4789,2023.1775



Scaled feature preview:


,Postal Code,Log_EV_Count,BEV_Percentage,Avg_Model_Year
0,98001,0.5953,1.2093,0.7586
1,98002,-0.5079,-1.3538,-0.6904
2,98003,0.4056,0.3673,0.1783
3,98004,1.9325,1.3488,0.9620
4,98005,1.1599,1.1169,1.7414


## 5. Prepare Association-Rule Transactions

Each vehicle transaction has a manufacturer, EV type (BEV or PHEV), and five-year model-year group.

CAFV eligibility is excluded from this first analysis. Keep vehicle IDs for tracing, but not as encoded features. True means an item is present; separate vehicle records remain separate transactions even when their items match.

In [5]:

association_data = df_wa[[
    "DOL Vehicle ID",
    "Make",
    "Electric Vehicle Type",
    "Model Year"
]].copy()


association_data["EV_Type"] = (
    association_data["Electric Vehicle Type"].map({
        "Battery Electric Vehicle (BEV)": "BEV",
        "Plug-in Hybrid Electric Vehicle (PHEV)": "PHEV"
    })
)

year_start = (association_data["Model Year"] // 5) * 5

association_data["Year_Group"] = (
    year_start.astype(str)
    + "-"
    + (year_start + 4).astype(str)
)

association_data = association_data[[
    "DOL Vehicle ID",
    "Make",
    "EV_Type",
    "Year_Group"
]].reset_index(drop=True)

association_features = ["Make", "EV_Type", "Year_Group"]

assert not association_data[association_features].isna().any().any()


association_encoded = pd.get_dummies(
    association_data[association_features],
    prefix_sep="=",
    dtype=bool
)


assert len(association_data) == len(df_wa)
assert association_data["DOL Vehicle ID"].is_unique
assert association_encoded.index.equals(association_data.index)
assert association_encoded.columns.is_unique


assert association_encoded.sum(axis=1).eq(3).all()


for feature in association_features:
    feature_columns = [
        column for column in association_encoded.columns
        if column.startswith(feature + "=")
    ]
    assert association_encoded[feature_columns].sum(axis=1).eq(1).all()


print("Vehicle transactions:", len(association_data))
print("Encoded matrix shape:", association_encoded.shape)
print(
    "Items per transaction:",
    sorted(association_encoded.sum(axis=1).unique().tolist())
)

print("\nNumber of categories per attribute:")
print(
    association_data[association_features]
    .nunique()
    .to_string()
)

print("\nTransaction preview:")
display(association_data.head())

print("\nFive-year group counts:")
print(
    association_data["Year_Group"]
    .value_counts()
    .sort_index()
    .to_string()
)


print("\nEncoded transaction examples:")
for row_number in range(5):
    present_items = association_encoded.columns[
        association_encoded.iloc[row_number].to_numpy()
    ].tolist()
    print(f"Vehicle {row_number + 1}: {present_items}")



Vehicle transactions: 49873
Encoded matrix shape: (49873, 56)
Items per transaction: [3]

Number of categories per attribute:
Make          47
EV_Type        2
Year_Group     7

Transaction preview:


,DOL Vehicle ID,Make,EV_Type,Year_Group
0,272203270,LEXUS,BEV,2020-2024
1,286868704,HYUNDAI,BEV,2025-2029
2,285893233,VOLKSWAGEN,BEV,2025-2029
3,252551757,RIVIAN,BEV,2020-2024
4,269998305,TESLA,BEV,2020-2024



Five-year group counts:
Year_Group
1995-1999        1
2000-2004        4
2005-2009        3
2010-2014     1491
2015-2019     7122
2020-2024    29012
2025-2029    12240

Encoded transaction examples:
Vehicle 1: ['Make=LEXUS', 'EV_Type=BEV', 'Year_Group=2020-2024']
Vehicle 2: ['Make=HYUNDAI', 'EV_Type=BEV', 'Year_Group=2025-2029']
Vehicle 3: ['Make=VOLKSWAGEN', 'EV_Type=BEV', 'Year_Group=2025-2029']
Vehicle 4: ['Make=RIVIAN', 'EV_Type=BEV', 'Year_Group=2020-2024']
Vehicle 5: ['Make=TESLA', 'EV_Type=BEV', 'Year_Group=2020-2024']


## 6. Prepare Anomaly-Detection Inputs

Use only vehicles with positive recorded range. Prepare BEVs and PHEVs separately, using model year and usable range as numeric inputs. Keep make, model, and vehicle ID for reviewing flagged records.

Keep positive values flagged by the earlier IQR check; they may be unusual, not errors. Isolation Forest is planned, so the numeric features stay in their original units. Results cover only vehicles with positive ranges.

In [6]:

import numpy as np
anomaly_data = df_wa.loc[
    df_wa["Electric Range Usable"].notna(),
    [
        "DOL Vehicle ID",
        "Make",
        "Model",
        "Electric Vehicle Type",
        "Model Year",
        "Electric Range Usable"
    ]
].copy().reset_index(drop=True)

anomaly_data["EV_Type"] = (
    anomaly_data["Electric Vehicle Type"].map({
        "Battery Electric Vehicle (BEV)": "BEV",
        "Plug-in Hybrid Electric Vehicle (PHEV)": "PHEV"
    })
)

anomaly_data = anomaly_data[[
    "DOL Vehicle ID",
    "Make",
    "Model",
    "EV_Type",
    "Model Year",
    "Electric Range Usable"
]]

anomaly_features = [
    "Model Year",
    "Electric Range Usable"
]

anomaly_inputs = {
    ev_type: anomaly_data.loc[
        anomaly_data["EV_Type"].eq(ev_type),
        anomaly_features
    ].copy()
    for ev_type in ["BEV", "PHEV"]
}


assert len(anomaly_data) == (
    df_wa["Electric Range Usable"].notna().sum()
)
assert anomaly_data["DOL Vehicle ID"].is_unique
assert anomaly_data["EV_Type"].isin(["BEV", "PHEV"]).all()
assert anomaly_data["Electric Range Usable"].gt(0).all()

assert sum(len(table) for table in anomaly_inputs.values()) == (
    len(anomaly_data)
)

for ev_type, table in anomaly_inputs.items():
    assert not table.empty
    assert list(table.columns) == anomaly_features
    assert np.isfinite(table.to_numpy()).all()
    assert anomaly_data.loc[
        table.index, "EV_Type"
    ].eq(ev_type).all()


availability = (
    df_wa.groupby("Electric Vehicle Type")
    .agg(
        Vehicle_Count=("DOL Vehicle ID", "size"),
        Positive_Range_Count=("Electric Range Usable", "count")
    )
)

availability["Excluded_Unavailable_Range"] = (
    availability["Vehicle_Count"]
    - availability["Positive_Range_Count"]
)

availability["Included_Percent"] = (
    availability["Positive_Range_Count"]
    / availability["Vehicle_Count"] * 100
)

print("Total anomaly records:", len(anomaly_data))

print("\nModel input shapes:")
for ev_type, table in anomaly_inputs.items():
    print(f"{ev_type}: {table.shape}")

print("\nCoverage by vehicle type:")
display(availability.round(2))

print("\nAnomaly record preview:")
display(anomaly_data.head())

print("\nNumeric summaries by vehicle type:")
display(
    anomaly_data.groupby("EV_Type")[anomaly_features]
    .agg(["min", "median", "max"])
    .round(2)
)



Total anomaly records: 17202

Model input shapes:
BEV: (7504, 2)
PHEV: (9698, 2)

Coverage by vehicle type:


,Vehicle_Count,Positive_Range_Count,Excluded_Unavailable_Range,Included_Percent
Electric Vehicle Type,,,,
Battery Electric Vehicle (BEV),40168,7504,32664,18.68
Plug-in Hybrid Electric Vehicle (PHEV),9705,9698,7,99.93



Anomaly record preview:


,DOL Vehicle ID,Make,Model,EV_Type,Model Year,Electric Range Usable
0,276243026,TESLA,MODEL Y,BEV,2020,291.0
1,476992141,TESLA,MODEL 3,BEV,2019,220.0
2,103214044,CHEVROLET,BOLT EV,BEV,2020,259.0
3,294698974,BMW,330E,PHEV,2021,20.0
4,269409304,TESLA,MODEL 3,BEV,2019,220.0



Numeric summaries by vehicle type:


Model Year               Electric Range Usable              
               min  median   max                   min median    max
EV_Type                                                             
BEV           1999  2018.0  2024                  57.0  215.0  337.0
PHEV          2010  2023.0  2027                   1.0   32.0  153.0

## 7. Proximity Choices and Planned Mining Methods

### 7.1 ZIP-code clustering
Question: What fleet profiles appear across ZIP codes?

Inputs: 249 ZIP codes with at least 30 sampled vehicles; standardized Log_EV_Count, BEV_Percentage, and Avg_Model_Year.

Method: K-means with Euclidean distance. Compare 2–6 clusters using random_state=42 and n_init=20. Review silhouette score, inertia, cluster sizes, and profiles in original units.

ZIP codes and coordinates are not distance inputs, so geographic neighbors may fall into different clusters. Results depend on features, scaling, and the cutoff.
### 7.2 Association-rule mining
Question: Which manufacturer, EV type, and model-year combinations occur together?

Inputs: 49,873 transactions, each with one manufacturer, one EV type, and one year group.

Method: Apriori and association rules. Start with support ≥ 1%, confidence ≥ 60%, and itemsets up to length 3; review rules with lift above 1.

Support measures how often items occur together, confidence how often B appears when A appears, and lift compares confidence with B's overall frequency. Lift above 1 shows positive association in this sample, not causation or statistical significance. Review counts and usefulness too. The support threshold can exclude rare combinations.
### 7.3 Vehicle-level anomaly detection
Question: Which vehicles have unusual range and model-year combinations within their EV type?

Inputs: 7,504 BEVs and 9,698 PHEVs with positive ranges; use model year and usable range.

Method: Separate Isolation Forest models for BEVs and PHEVs, with 200 trees, max_samples=256, contamination="auto", and random_state=42. Review roughly the top 1% by anomaly score.

Inspect flagged records using make, model, and year, and compare them with earlier IQR flags. Flags are not proof of errors; without verified labels, accuracy cannot be measured.







### Justification of Preparation and Proximity Choices

- Keep ZIP codes with at least 30 sampled vehicles to reduce unstable summaries from small groups.
- Log-transform EV counts to reduce skew, then standardize clustering features to comparable scales.
- Keep range summaries outside clustering because range availability differs by vehicle type and model year.
- Use Euclidean distance to compare standardized ZIP profiles.
- Association rules use item co-occurrence, so no pairwise distance is required. Five-year groups follow Week 6.
- Exclude CAFV eligibility to focus association rules on manufacturer, vehicle type, and model-year group.
- Isolation Forest uses random splits rather than distance. Keep numeric inputs in original units and analyze BEVs and PHEVs separately.
- The top 1% is a review budget, not an estimated error rate. Range-based findings apply only to records with positive ranges.

Justification of model settings

K-means, 2 to 6 clusters, n_init=20: with only 249 ZIP codes, more clusters would leave very small groups. Repeated starts reduce dependence on the random start.

Support >= 1%: with 49,873 transactions, an itemset must cover about 500 vehicles, which avoids rules built on rare combinations.

Confidence >= 60% and itemsets up to length 3: a moderate starting threshold, with lift above 1 as the main filter. There are only three attributes, so three is the maximum itemset length.

200 trees and max_samples=256: 256 is the subsample size suggested in the original Isolation Forest paper. 200 trees gives more stable scores than the default 100.

Top 1%: a review budget, about 75 BEVs and 97 PHEVs, not an estimated error rate.

These are starting values, not tuned results.

## 8. Save the prepared datasets

In [7]:
import hashlib
import json
import platform
import sklearn
from pandas.testing import assert_frame_equal

OUTPUT_DIR = Path("data/processed/week09")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tables_to_save = {
    "zip_summary_all.csv": zip_summary,
    "zip_clustering_profiles.csv": zip_prepared,
    "zip_clustering_scaled.csv": zip_scaled,
    "association_transactions.csv": association_data,
    "association_encoded.csv": association_encoded,
    "anomaly_vehicles.csv": anomaly_data
}

metadata = {
    "source_sample": DATA_PATH.as_posix(),
    "source_sha256": hashlib.sha256(
        DATA_PATH.read_bytes()
    ).hexdigest(),
    "sample_seed": 42,
    "sample_rows": int(len(df)),
    "washington_rows": int(len(df_wa)),
    "minimum_sampled_vehicles_per_zip": int(MIN_ZIP_VEHICLES),
    "clustering_features": cluster_features,
    "count_transformation": "natural log of (1 + EV_Count)",
    "scaling": {
        "method": "StandardScaler fitted on retained ZIP codes",
        "feature_order": cluster_features,
        "mean": cluster_scaler.mean_.tolist(),
        "scale": cluster_scaler.scale_.tolist()
    },
    "association_features": association_features,
    "year_group_rule": "floor(Model Year / 5) * 5 through start + 4",
    "association_row_alignment": (
        "Encoded rows match association_transactions.csv in order"
    ),
    "anomaly_features": anomaly_features,
    "anomaly_grouping": "Separate BEV and PHEV inputs",
    "range_policy": "Only positive recorded ranges are usable",
    "software": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__
    }
}


file_contents = {
    name: table.to_csv(index=False, lineterminator="\n").encode("utf-8")
    for name, table in tables_to_save.items()
}

file_contents["preparation_metadata.json"] = (
    json.dumps(metadata, indent=2, allow_nan=False) + "\n"
).encode("utf-8")


conflicts = [
    name
    for name, content in file_contents.items()
    if (OUTPUT_DIR / name).exists()
    and (OUTPUT_DIR / name).read_bytes() != content
]

if conflicts:
    raise FileExistsError(
        "Existing files have different contents. Nothing was overwritten. "
        "Share this message before continuing: "
        + ", ".join(conflicts)
    )


for name, content in file_contents.items():
    path = OUTPUT_DIR / name
    if not path.exists():
        with path.open("xb") as file:
            file.write(content)


verification_rows = []

for name, original in tables_to_save.items():
    read_options = {}

    if "Postal Code" in original.columns:
        read_options["dtype"] = {"Postal Code": "string"}

    reloaded = pd.read_csv(
        OUTPUT_DIR / name,
        **read_options
    )

    assert_frame_equal(
        original.reset_index(drop=True),
        reloaded.reset_index(drop=True),
        check_dtype=False,
        check_exact=False,
        rtol=1e-10,
        atol=1e-10
    )

    verification_rows.append({
        "File": name,
        "Rows": len(reloaded),
        "Columns": len(reloaded.columns),
        "Reload_Check": "Passed"
    })


with (OUTPUT_DIR / "preparation_metadata.json").open(
    encoding="utf-8"
) as file:
    assert json.load(file) == metadata

display(pd.DataFrame(verification_rows))


print("Output folder:", OUTPUT_DIR.resolve())


,File,Rows,Columns,Reload_Check
0,zip_summary_all.csv,509,10,Passed
1,zip_clustering_profiles.csv,249,11,Passed
2,zip_clustering_scaled.csv,249,4,Passed
3,association_transactions.csv,49873,4,Passed
4,association_encoded.csv,49873,56,Passed
5,anomaly_vehicles.csv,17202,6,Passed


Output folder: C:\Users\Asus\Documents\Data Mining Project F26\data\processed\week09
